In [ ]:
!pip install pysam

In [ ]:
from tqdm import tqdm
import pysam

In [ ]:
b = 0

In [ ]:
inds_sibs = {}

with open(f'/mnt/project/DNM/sibs/blocks/sibs_pairs_b{b}.txt') as F:
            
    for i, row in enumerate(F):
                
        row = row.strip()
        row = row.split(' ')
    
        sib1 = str(row[0])
        sib2 = str(row[1])
            
        sib1_idx = f'{sib1}_b{b}_p{i}'
        sib2_idx = f'{sib2}_b{b}_p{i}'
            
        inds_sibs[sib1_idx] = sib2_idx
        inds_sibs[sib2_idx] = sib1_idx

In [ ]:
inds_diffs = {}

for chrom in range(1, 23):

    path = f'/mnt/project/DNM/sibs/diffs/chr{chrom}/'

    with open(f'/mnt/project/DNM/sibs/blocks/sibs_pairs_b{b}.txt') as F:

        for i, _ in enumerate(F):

            # Remove pair if sib is missing
            cnt = 0

            with open(f'{path}sibs_chr{chrom}_b{b}_p{i}.tsv') as f:

                for row in f:

                    row = row.strip()
                    row = row.split()
                    cnt += 1 if len(row) > 0 else 0

            if cnt < 2:
                continue

            # Record differences per sib
            with open(f'{path}sibs_chr{chrom}_b{b}_p{i}.tsv') as f:

                for row in f:

                    row = row.strip()
                    row = row.split()

                    ind = str(row[0].split('_')[1])
                    ind_idx = f'{ind}_b{b}_p{i}'
                    if ind_idx not in inds_diffs:
                        inds_diffs[ind_idx] = []

                    ind_diffs = row[1].split('|')[1:] if len(row) == 2 else []

                    for ind_diff in ind_diffs:
                        inds_diffs[ind_idx].append(ind_diff)

In [ ]:
inds_path = {}

# paths = '/mnt/project/Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]/'

for ind_idx in inds_sibs:
    
    ind = str(ind_idx.split('_')[0])
    # inds_path[ind_idx] = f'{paths}{ind[:2]}/{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'
    inds_path[ind_idx] = f'{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'

In [ ]:
inds_diffs_qc = {}
inds_diffs_errors = {}

for _, ind_idx in tqdm(enumerate(inds_diffs)):
    
    # Sib --------------------------------
    
    sib_idx = inds_sibs[ind_idx]
    
    try:
        vcf = pysam.VariantFile(inds_path[sib_idx])
    except:
        continue # failed to read gVCF

    sib_info = {}
    
    for diff_id in inds_diffs[ind_idx]:
        
        chrom = int(diff_id.split(':')[0])
        pos = int(diff_id.split(':')[1])
        ref = diff_id.split(':')[2]
        alt = diff_id.split(':')[3]
            
        for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):
            
            for f in list(F.samples):
            
                dp = F.samples[f]['DP']
                gq = F.samples[f]['GQ']
                gt = F.samples[f]['GT']
                
                if None in gt:
                    continue
                
                if F.pos == pos:
                    
                    ref_ad = 0
                    alt_ad = 0
                    
                    if ref in F.alleles:
                        i = F.alleles.index(ref)
                        ref_ad = F.samples[f]['AD'][i]
                        
                    if alt in F.alleles:
                        i = F.alleles.index(alt)
                        alt_ad = F.samples[f]['AD'][i]
                        
                    alleles = (F.alleles[gt[0]], F.alleles[gt[1]])
                        
                else:
                    ref_ad = dp
                    alt_ad = 0
                    alleles = (ref, ref)
                
                sib_info[diff_id] = (ref_ad, alt_ad, dp, gq, alleles)
        
    # Individual --------------------------------
    
    try:
        vcf = pysam.VariantFile(inds_path[ind_idx])
    except:
        continue # failed to read gVCF
    
    inds_diffs_qc[ind_idx] = []
    inds_diffs_errors[ind_idx] = []
    
    for diff_id in inds_diffs[ind_idx]:

        if diff_id not in sib_info:
            continue
            
        chrom = int(diff_id.split(':')[0])
        pos = int(diff_id.split(':')[1])
        ref = diff_id.split(':')[2]
        alt = diff_id.split(':')[3]
            
        for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):
            
            for f in list(F.samples):
            
                dp = F.samples[f]['DP']
                gq = F.samples[f]['GQ']
                gt = F.samples[f]['GT']
                
                if None in gt:
                    continue
                
                if F.pos == pos:
                    
                    ref_ad = 0
                    alt_ad = 0
                    
                    if ref in F.alleles:
                        i = F.alleles.index(ref)
                        ref_ad = F.samples[f]['AD'][i]
                        
                    if alt in F.alleles:
                        i = F.alleles.index(alt)
                        alt_ad = F.samples[f]['AD'][i]
                        
                    alleles = (F.alleles[gt[0]], F.alleles[gt[1]])
                        
                else:
                    ref_ad = dp
                    alt_ad = 0
                    alleles = (ref, ref)
                    
                info = (diff_id, ref_ad, alt_ad, dp, gq, alleles,
                        sib_info[diff_id][0], sib_info[diff_id][1], sib_info[diff_id][2], sib_info[diff_id][3], sib_info[diff_id][4])
                        
                if ((ref in alleles and alt in alleles) and
                    (sib_info[diff_id][4][0] == sib_info[diff_id][4][1] and ref in sib_info[diff_id][4])):
                    inds_diffs_qc[ind_idx].append(info)
                else:
                    inds_diffs_errors[ind_idx].append(info)

In [ ]:
with open(f'sibs_qc_b{b}.txt', 'w') as f:
    for ind_idx in inds_diffs_qc:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, sib_ref_ad, sib_alt_ad, sib_dp, sib_gq, sib_alleles) in inds_diffs_qc[ind_idx]:
            f.write(f'{ind_idx} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {sib_ref_ad} {sib_alt_ad} {sib_dp} {sib_gq} {sib_alleles}\n')

In [ ]:
with open(f'sibs_errors_b{b}.txt', 'w') as f:
    for ind_idx in inds_diffs_errors:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, sib_ref_ad, sib_alt_ad, sib_dp, sib_gq, sib_alleles) in inds_diffs_errors[ind_idx]:
            f.write(f'{ind_idx} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {sib_ref_ad} {sib_alt_ad} {sib_dp} {sib_gq} {sib_alleles}\n')